# Week 2: Inference and Sampling

This notebook opens the two stages behind text generation:

1. the **forward pass**, which turns a prompt into next-token logits and probabilities, and  
2. the **sampling step**, which reshapes that distribution and chooses from it.

All numerical measurements in Parts 1–4 are computed from a local `distilgpt2` model. No forward-pass values are hard-coded.

> **AI-use note:** Gemini may be used for scaffolding or debugging, but it should not supply the measurements or conclusions in this notebook. The assignment requires those to come from the model run and from my own interpretation.


In [ ]:
# Setup
# In Google Colab, uncomment the next line on the first run:
# !pip install -q transformers torch numpy matplotlib

import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_HOME"] = str((pathlib.Path(".") / ".hf_cache").resolve())
torch.manual_seed(0)

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",   # required so attention weights are returned
)
model.eval()

cfg = model.config

print(
    f"{MODEL_NAME}: "
    f"{cfg.n_layer} layers, "
    f"{cfg.n_head} heads, "
    f"embedding dimension {cfg.n_embd}, "
    f"vocabulary size {cfg.vocab_size}"
)


## Part 1: Trace the forward pass

I use the short prompt below and trace it through tokenization, token embeddings, one attention head, and the final vocabulary logits. Every reported number is produced by the model during this run.


In [ ]:
prompt = "Artificial intelligence can help students"

encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"][0]

print("Prompt:", repr(prompt))
print("Token IDs:", input_ids.tolist())
print("\nTokenization:")
for position, token_id in enumerate(input_ids.tolist()):
    print(
        f"  position {position:2d}: "
        f"id {token_id:6d} -> {tokenizer.decode([token_id])!r}"
    )

# Token embedding lookup.
embeddings = model.transformer.wte(input_ids)

print("\nEmbedding dimension:", cfg.n_embd)
print("Embedded prompt shape (sequence length, embedding dimension):",
      tuple(embeddings.shape))

# One forward pass with attention weights requested.
with torch.no_grad():
    outputs = model(**encoded, output_attentions=True)

# Shape of one attention tensor:
# (batch, heads, query_positions, key_positions)
layer_index = 0
head_index = 0
query_position = input_ids.shape[0] - 1

attention = outputs.attentions[layer_index][
    0, head_index, query_position
]

print(
    f"\nAttention weights for the LAST prompt token "
    f"at layer {layer_index}, head {head_index}:"
)
print("Sum over allowed positions:", attention.sum().item())

for key_position in range(input_ids.shape[0]):
    token_text = tokenizer.decode([input_ids[key_position]])
    print(
        f"  attends to position {key_position:2d} "
        f"{token_text!r:18s}: {attention[key_position].item():.6f}"
    )

logits = outputs.logits
next_token_logits = logits[0, -1]
next_token_probs = torch.softmax(next_token_logits, dim=-1)

print("\nLogits shape (batch, sequence length, vocabulary):", tuple(logits.shape))
print("Vocabulary size from config:", cfg.vocab_size)
print("Last-position logits shape:", tuple(next_token_logits.shape))

top10 = torch.topk(next_token_probs, 10)

print("\nTop 10 next-token probabilities:")
for rank, (probability, token_id) in enumerate(
    zip(top10.values.tolist(), top10.indices.tolist()), start=1
):
    print(
        f"{rank:2d}. id {token_id:6d} "
        f"{tokenizer.decode([token_id])!r:18s} "
        f"p={probability:.8f}"
    )


### Forward-pass interpretation

The prompt first becomes discrete token IDs. The embedding table maps each ID into a vector with the model's embedding dimension. Inside the transformer, causal self-attention assigns weights only to positions the current token is allowed to see; the weights shown above sum to approximately 1. The final hidden state is projected to one logit for every vocabulary item. Softmax converts the last-position logits into the next-token probability distribution shown by the top-ten list.


## Part 2: Build the sampling explorer

The functions below implement the three sampling controls directly.

- **Temperature** rescales logits before softmax. A temperature below 1 sharpens the distribution; a temperature above 1 flattens it.
- **Top-k** sets every probability outside the `k` highest-probability tokens to zero, then renormalizes.
- **Top-p** sorts tokens from most to least probable and keeps the smallest prefix whose cumulative probability reaches the requested threshold, then renormalizes.

For combinations, I apply controls in the common inference order:

**temperature → top-k → top-p**.


In [ ]:
# Convert the real next-token logits to NumPy for the sampling explorer.
z = next_token_logits.detach().cpu().numpy()

def softmax(logits):
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()

def apply_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    return softmax(np.asarray(logits, dtype=np.float64) / temperature)

def apply_top_k(probabilities, k):
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if k is None:
        return probabilities / probabilities.sum()

    if not 1 <= k <= len(probabilities):
        raise ValueError(f"k must be between 1 and {len(probabilities)}")

    keep = np.argpartition(probabilities, -k)[-k:]
    filtered = np.zeros_like(probabilities)
    filtered[keep] = probabilities[keep]

    return filtered / filtered.sum()

def apply_top_p(probabilities, p):
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if not 0 < p <= 1:
        raise ValueError("top-p must satisfy 0 < p <= 1")

    # By definition, p=1.0 performs no nucleus truncation.
    if p == 1.0:
        return probabilities / probabilities.sum()

    order = np.argsort(probabilities)[::-1]
    sorted_probs = probabilities[order]
    cumulative = np.cumsum(sorted_probs)

    # Keep the smallest prefix whose cumulative probability reaches p.
    cut = np.searchsorted(cumulative, p, side="left") + 1
    keep = order[:cut]

    filtered = np.zeros_like(probabilities)
    filtered[keep] = probabilities[keep]

    return filtered / filtered.sum()

def apply_sampling(logits, temperature=1.0, top_k=None, top_p=1.0):
    probabilities = apply_temperature(logits, temperature)
    probabilities = apply_top_k(probabilities, top_k)
    probabilities = apply_top_p(probabilities, top_p)
    return probabilities

def surviving_tokens(probabilities):
    return int(np.count_nonzero(probabilities > 0))

def entropy(probabilities):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    nonzero = probabilities[probabilities > 0]
    return float(-(nonzero * np.log(nonzero)).sum())

def plot_top_probabilities(probabilities, title, n=15):
    order = np.argsort(probabilities)[::-1][:n]
    labels = [tokenizer.decode([int(i)]).replace("\n", "\\n") for i in order]
    values = probabilities[order]

    plt.figure(figsize=(10, 4))
    plt.bar(range(len(values)), values)
    plt.xticks(range(len(values)), labels, rotation=60, ha="right")
    plt.ylabel("Probability")
    plt.title(title)
    plt.tight_layout()
    plt.show()

base = softmax(z)

print("Baseline distribution")
print("  entropy:", round(entropy(base), 6))
print("  maximum probability:", round(float(base.max()), 6))
print("  surviving tokens:", surviving_tokens(base))


### 2A. Temperature: before and after


In [ ]:
plot_top_probabilities(base, "Before temperature scaling: T = 1.0")

temp_low = apply_temperature(z, 0.5)
plot_top_probabilities(temp_low, "After temperature scaling: T = 0.5")

temp_high = apply_temperature(z, 1.5)
plot_top_probabilities(temp_high, "After temperature scaling: T = 1.5")

print("Maximum probability")
print("  baseline T=1.0:", round(float(base.max()), 6))
print("  T=0.5:", round(float(temp_low.max()), 6))
print("  T=1.5:", round(float(temp_high.max()), 6))


### 2B. Top-k: before and after


In [ ]:
plot_top_probabilities(base, "Before top-k filtering")

topk_20 = apply_top_k(base, 20)
plot_top_probabilities(topk_20, "After top-k filtering: k = 20")

print("Surviving tokens before top-k:", surviving_tokens(base))
print("Surviving tokens after top-k=20:", surviving_tokens(topk_20))
print("Maximum probability after top-k=20:", round(float(topk_20.max()), 6))


### 2C. Top-p: before and after


In [ ]:
plot_top_probabilities(base, "Before top-p filtering")

topp_090 = apply_top_p(base, 0.90)
plot_top_probabilities(topp_090, "After top-p filtering: p = 0.90")

print("Surviving tokens before top-p:", surviving_tokens(base))
print("Surviving tokens after top-p=0.90:", surviving_tokens(topp_090))
print("Maximum probability after top-p=0.90:", round(float(topp_090.max()), 6))


### 2D. Combination 1: temperature + top-k


In [ ]:
combo_1 = apply_sampling(
    z,
    temperature=0.7,
    top_k=40,
    top_p=1.0,
)

plot_top_probabilities(
    combo_1,
    "Combination 1: temperature = 0.7, top-k = 40"
)

print("Combination 1")
print("  entropy:", round(entropy(combo_1), 6))
print("  maximum probability:", round(float(combo_1.max()), 6))
print("  surviving tokens:", surviving_tokens(combo_1))


### 2E. Combination 2: temperature + top-k + top-p


In [ ]:
combo_2 = apply_sampling(
    z,
    temperature=1.2,
    top_k=100,
    top_p=0.90,
)

plot_top_probabilities(
    combo_2,
    "Combination 2: temperature = 1.2, top-k = 100, top-p = 0.90"
)

print("Combination 2")
print("  entropy:", round(entropy(combo_2), 6))
print("  maximum probability:", round(float(combo_2.max()), 6))
print("  surviving tokens:", surviving_tokens(combo_2))


## Part 3: Prediction vs. measured result

I chose three settings that collectively exercise temperature, top-k, and top-p. The third setting combines all three controls.

### Predictions

**Setting A — temperature 0.5, no truncation.**  
I expect the lower temperature to concentrate more probability on the already likely tokens. The maximum probability should increase and entropy should decrease. Because there is no top-k or top-p truncation, all vocabulary tokens should remain available.

**Setting B — temperature 1.0, top-k 20.**  
I expect exactly 20 tokens to survive. Renormalizing the distribution over only those 20 tokens should increase the maximum probability and reduce entropy compared with the baseline.

**Setting C — temperature 1.2, top-k 100, top-p 0.90.**  
Temperature 1.2 should initially flatten the distribution, but top-k then removes everything outside the 100 highest-probability tokens. Top-p can reduce that set further if fewer than 100 tokens are needed to reach 90% cumulative probability. I therefore expect no more than 100 tokens to survive.


In [ ]:
settings = [
    {
        "name": "A",
        "temperature": 0.5,
        "top_k": None,
        "top_p": 1.0,
        "prediction": (
            "Lower temperature should sharpen the distribution: "
            "lower entropy, higher max probability, all tokens survive."
        ),
    },
    {
        "name": "B",
        "temperature": 1.0,
        "top_k": 20,
        "top_p": 1.0,
        "prediction": (
            "Top-k should keep exactly 20 tokens and renormalization "
            "should raise the maximum probability."
        ),
    },
    {
        "name": "C",
        "temperature": 1.2,
        "top_k": 100,
        "top_p": 0.90,
        "prediction": (
            "Higher temperature first flattens the distribution, but "
            "top-k limits it to at most 100 tokens and top-p may cut it further."
        ),
    },
]

baseline_entropy = entropy(base)
baseline_max = float(base.max())
baseline_survivors = surviving_tokens(base)

print(
    f"Baseline: entropy={baseline_entropy:.6f}, "
    f"max_probability={baseline_max:.6f}, "
    f"survivors={baseline_survivors}\n"
)

results = {}

for setting in settings:
    p = apply_sampling(
        z,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        top_p=setting["top_p"],
    )

    result = {
        "entropy": entropy(p),
        "max_probability": float(p.max()),
        "survivors": surviving_tokens(p),
    }
    results[setting["name"]] = result

    print(f"Setting {setting['name']}")
    print(
        f"  parameters: temperature={setting['temperature']}, "
        f"top_k={setting['top_k']}, top_p={setting['top_p']}"
    )
    print("  prediction:", setting["prediction"])
    print(f"  measured entropy: {result['entropy']:.6f}")
    print(f"  measured maximum probability: {result['max_probability']:.6f}")
    print(f"  measured surviving-token count: {result['survivors']}")
    print()


In [ ]:
# Mechanistic comparison of predictions to measurements.
a = results["A"]
b = results["B"]
c = results["C"]

print("Prediction vs. measured result\n")

print("Setting A:")
print(
    "  Prediction matched:",
    a["entropy"] < baseline_entropy
    and a["max_probability"] > baseline_max
    and a["survivors"] == baseline_survivors,
)
print(
    f"  Entropy changed from {baseline_entropy:.6f} to {a['entropy']:.6f}; "
    f"max probability changed from {baseline_max:.6f} to {a['max_probability']:.6f}; "
    f"{a['survivors']} tokens survived."
)

print("\nSetting B:")
print(
    "  Prediction matched:",
    b["survivors"] == 20
    and b["max_probability"] > baseline_max,
)
print(
    f"  Top-k removed all but {b['survivors']} tokens. "
    f"After renormalization, max probability became {b['max_probability']:.6f} "
    f"and entropy became {b['entropy']:.6f}."
)

print("\nSetting C:")
print(
    "  Prediction matched:",
    c["survivors"] <= 100,
)
print(
    f"  The final distribution kept {c['survivors']} tokens, "
    f"with entropy {c['entropy']:.6f} and "
    f"maximum probability {c['max_probability']:.6f}."
)
print(
    "  Mechanistically, temperature changed the relative logit gaps first; "
    "top-k then imposed a hard 100-token ceiling; top-p then retained only "
    "the smallest remaining prefix needed to reach 0.90 cumulative probability."
)


## Part 4: Failure case — `top-p = 1.0` keeps the full vocabulary

A newcomer might assume that top-p always removes low-probability tokens. That is not true when `p = 1.0`.

Top-p retains the smallest high-probability prefix whose cumulative probability reaches the threshold. A threshold of 1.0 means the retained set must account for 100% of the probability mass. Because softmax assigns a positive probability to every vocabulary item, the full vocabulary survives.

**Mitigation:** use a threshold below 1.0 when nucleus truncation is actually desired, and verify the surviving-token count rather than assuming top-p performed a cut.


In [ ]:
failure = apply_top_p(base, 1.0)

print("Failure case: top-p = 1.0")
print("Vocabulary size:", cfg.vocab_size)
print("Surviving tokens:", surviving_tokens(failure))
print("Distribution unchanged:", np.allclose(base, failure))
print("Maximum probability before:", round(float(base.max()), 8))
print("Maximum probability after :", round(float(failure.max()), 8))

assert surviving_tokens(failure) == cfg.vocab_size
assert np.allclose(base, failure)


## Part 5: Submission checklist

Before exporting and submitting:

- Run the notebook **top to bottom** so every figure and measured value is visible.
- Confirm Part 1 shows token IDs, embedding dimension, one attention vector that sums to approximately 1, logits shape/vocabulary size, and top-ten next-token probabilities.
- Confirm Part 2 shows before/after figures for temperature, top-k, and top-p plus at least two combinations.
- Confirm Part 3 contains three predictions and the measured entropy, maximum probability, and surviving-token count for all three settings.
- Confirm Part 4 visibly demonstrates the failure case, explains the cause, and states a mitigation.
- Export the executed notebook to **PDF or HTML** for Canvas.
- Commit the same notebook to the course repository.
- Open a pull request with a concise result summary and a link to the required research-note issue.
- Record any AI assistance in the repository's agent context file. If Gemini API was used for scaffolding/debugging, state exactly what it helped with and make clear that the numerical measurements came from the local model.
